# TaxPulse — Income Event Export Exploratory Data Analysis (EDA)

**Objective**: Profile the full-scale vendor income event export (`data/exports/income_events.csv.gz`), identify inference traps and data anomalies, verify dtype discipline, and benchmark eager vs lazy performance.

In [1]:
import gzip
import time
import tracemalloc
import polars as pl
import pandas as pd
import duckdb

EXPORT_PATH = "../data/exports/income_events.csv.gz"

## 1. Raw Text Inspection
Inspect raw text lines before running any inference engine to observe original string representations (specifically leading zeros on identifiers).

In [2]:
with gzip.open(EXPORT_PATH, "rt") as f:
    for i in range(10):
        print(f.readline().strip())

## 2. Default Inferred Schema vs Declared Schema
Observe how default schema inference fails by converting `jurisdiction_code` to `Int64` (dropping leading zeros) and `effective_rate` to `Float64`.

In [3]:
# Default Inferred Load
df_inferred = pl.read_csv(EXPORT_PATH)
print("=== Inferred Schema ===")
for col, dtype in df_inferred.schema.items():
    null_cnt = df_inferred[col].null_count()
    print(f"{col:20s}: {dtype!s:15s} | nulls: {null_cnt:,} ({null_cnt/len(df_inferred)*100:.2f}%)")

## 3. Anomaly & Distribution Analysis

In [4]:
print("Total Rows:", len(df_inferred))
print("Min Amount (cents):", df_inferred["amount_cents"].min())
print("Max Amount (cents):", df_inferred["amount_cents"].max())
print("Negative Adjustments Count:", (df_inferred["amount_cents"] < 0).sum())
print("High-Net-Worth Outliers (>$250k):
", (df_inferred["amount_cents"] > 25_000_000).sum())
print("Distinct Clients:", df_inferred["client_id"].n_unique())
print("Distinct Planning Periods:", df_inferred["planning_period"].n_unique())

## 4. Declared Schema Load (Dtype Discipline)
Load with explicit schema overrides ensuring identifiers are strings and dates are datetimes.

In [5]:
from services.pipeline.schema import EXPORT_SCHEMA

df_declared = pl.read_csv(EXPORT_PATH, schema_overrides=EXPORT_SCHEMA)
print("=== Declared Schema ===")
for col, dtype in df_declared.schema.items():
    print(f"{col:20s}: {dtype!s:15s}")

# Verify leading zeros preserved
sample_jur = df_declared["jurisdiction_code"].filter(df_declared["jurisdiction_code"].str.starts_with("0")).head(5)
print("\nSample preserved leading zeros:", sample_jur.to_list())

## 5. Eager vs Lazy Performance Benchmark

In [6]:
# Polars Lazy aggregation
t0 = time.perf_counter()
q = (
    pl.scan_csv(EXPORT_PATH, schema_overrides=EXPORT_SCHEMA)
    .group_by("planning_period")
    .agg(pl.col("amount_cents").sum().alias("total_cents"))
    .sort("planning_period")
)
print(q.explain())
result_lazy = q.collect()
t_pl = time.perf_counter() - t0
print(f"Polars Lazy Execution Time: {t_pl:.3f}s")
print(result_lazy)